In [9]:


using Gen
using Plots
using Statistics
using Distributions 
using LinearAlgebra



function compute_rhat(chains)
    m = length(chains)   # Number of chains
    n = length(chains[1])  # Number of samples per chain
   
    # Compute chain means
    chain_means = [mean(chain) for chain in chains]
    grand_mean = mean(chain_means)

    # Compute within-chain variance W
    W = mean([var(chain, corrected=true) for chain in chains])  # corrected=true uses n-1 in denominator
    #println("W", W)
    # Compute between-chain variance B
    B = (n / (m - 1)) * sum((chain_mean - grand_mean)^2 for chain_mean in chain_means)
    #println("B", B)
    # Compute potential scale reduction factor (R-hat)
    var_hat = ((n - 1) / n) * W + (B / n)
    r_hat = sqrt(var_hat / W)
    #println(var_hat)
    #println(r_hat)
    return r_hat
end






@gen function linear_regression_model(X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma)
    # Sample the prior for intercept alpha and regression coefficients beta
    alpha ~ normal(0, sqrt(sigma_alpha2))            # Intercept
    beta = [{(:beta, i)} ~ normal(mu_beta, sqrt(sigma_beta2)) for i in 1:K]      # Regression coefficients
    
    # Sample degrees of freedom and scale for the Student's t-distribution
    nu ~ gamma(2, 10)                                # Degrees of freedom for Student's t
    sigma ~ exponential(lambda_sigma)                 # Scale for the Student's t

    # Compute the mean for each y_i: μ_i = α + X_i * β
    for i in 1:N
       
        mu_i = alpha + dot(X[i, :], beta)           # Linear model: μ_i = α + X_i * β
        {(:y, i)} ~ normal(mu_i, sigma)           # Sample y_i from Student's t distribution
    end
    
    return :y  # Return the observed responses
end


function multi_variable_metropolis(trace, model, (X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma), observations, eps)
    # Extract current values of parameters
    alpha_current = get_choices(trace)[:alpha]
    beta_current = [get_choices(trace)[(:beta, i)] for i in 1:length(K)]
    nu_current = get_choices(trace)[:nu]
    sigma_current = get_choices(trace)[:sigma]
    
    
    
    # Propose new values for mu, tau, and eta
    alpha_proposed = alpha_current + eps * randn()
    nu_proposed = nu_current +  eps *  randn()
    beta_proposed = [beta_current[i] + eps*randn() for i in 1:length(K)]
    log_sigma_proposed = log(sigma_current) + eps * randn()
    sigma_proposed = exp(log_sigma_proposed)
    
     # Create a temporary choice map with the proposed values
    temp_cm = choicemap(
        (:alpha => alpha_proposed),
        (:nu => nu_proposed),
        (:sigma => sigma_proposed)
        
    )
   
    # Add proposed eta values to the choice map
    for i in 1:length(K)
        
        temp_cm[(:beta, i)] = beta_proposed[i]
        # print("hoi") 
    end

    # Use Gen.update to get the updated trace after applying the proposed values
    (proposed_trace, _, _) = Gen.update(
        trace,                # Current trace                # Generative model
        (X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma), (),            # Arguments for the generative function
        temp_cm               # Temporary choice map with proposed values
    )

    ###
    current_score = get_score(trace)
  
    
    proposed_score = get_score(proposed_trace)
    # print(proposed_score)
    # Compute the acceptance ratio using the gradients
    acceptance_ratio = min(1.0, exp(proposed_score - current_score))
    # print(acceptance_ratio)
    # Accept or reject based on the acceptance ratio
    if rand() < acceptance_ratio
        return (proposed_trace, 1)
    else
        return (trace, 0)  # Keep the current state if not accepted
    end
end

function do_inference(model,  (X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma), y_obs, num_iters, eps)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end

     (trace, _) = generate(linear_regression_model, (X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma), observations)
    accepted = 0
    alpha_samples = []
    nu_samples = []

    # Store sampled values at each iteration
    for _ in 1:num_iters
        (trace, accepted_this_iter) = multi_variable_metropolis(trace, model, (X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma), observations, eps)
        accepted += accepted_this_iter
        
        # Store samples
        final_choices = get_choices(trace)
        push!(alpha_samples, final_choices[:alpha])
        push!(nu_samples, final_choices[:nu])
    end
    #print("hoi")
    acceptance_rate = accepted / num_iters  # Compute acceptance rate

    return (alpha_samples, nu_samples, acceptance_rate)
end





N = 100  # Number of data points
K = 5    # Number of features
sigma_alpha2 = 1.0  # Prior variance for alpha
mu_beta = 0.0  # Mean for beta prior
sigma_beta2 = 1.0  # Variance for beta prior
lambda_sigma = 1.0  # Rate for sigma prior

# Generate synthetic data for testing
X = randn(N, K)  # N x K feature matrix
true_alpha = 2.0
true_beta = randn(K)  # True regression coefficients
nu_true = 5.0
sigma_true = 1.0

# Linear model for generating synthetic y values
mu = X * true_beta .+ true_alpha
y = randn(N) .* sigma_true .+ mu  # Add noise to y values

num_iters = 2000  # Number of iterations for Metropolis-Hastings
eps = 0.1         # Step size for proposals
num_chains = 20
for eps in [0.1, 0.5, 0.9, 1, 1.1, 1.2, 1.5, 1.7, 2, 7]
    chains_mu = []
    chains_tau = []
    acceptance_rates = []
    for _ in 1:num_chains
        mu_samples, tau_samples, acc = do_inference(linear_regression_model,  (X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma), y, num_iters, eps)
        push!(chains_mu, mu_samples)
        push!(chains_tau, tau_samples)
        
        push!(acceptance_rates, acc)
    end
    println("Eps: ", eps)
    rhat_mu = compute_rhat(chains_mu)
    rhat_tau = compute_rhat(chains_tau)
    println("R-hat for alpha: ", rhat_mu)
    println("R-hat for nu: ", rhat_tau)
    means_per_chain = [mean(chain) for chain in chains_mu]
    println("Mean of alpha: ", mean(means_per_chain))
    means_per_chain_tau = [mean(chain) for chain in chains_tau]
    println("Mean of nu: ", mean(means_per_chain_tau))
    println("acceptance rate: ", mean(acceptance_rates))
    
end



Eps: 0.1
1.111893761005267
R-hat for nu: 13.231545821298903
Mean of alpha: 1.8840614694084163
17.902572793843326
acceptance rate: 0.5305500000000001
0.5: 
R-hat for alpha: 1.1367430042045867
R-hat for nu: 7.291088973104676
Mean of alpha: 2.0673610736953685
Mean of nu: 19.68978054756161
acceptance rate: 0.068875
0.9: 
R-hat for alpha: 1.3067513003840387
R-hat for nu: 7.364435460480013
Mean of alpha: 1.9180728477180597
Mean of nu: 20.891663370400394
acceptance rate: 0.017075000000000003
Eps: 1.0
R-hat for alpha: 1.2346072894807023
R-hat for nu: 5.745912476616501
Mean of alpha: 1.8607246718024686
Mean of nu: 18.544041106728976
acceptance rate: 0.016800000000000002
1.1: 
R-hat for alpha: 1.1938451399142063
R-hat for nu: 6.830458822561062
Mean of alpha: 1.9587374639810111
Mean of nu: 22.233831797456926
acceptance rate: 0.012925000000000002
1.2: 
R-hat for alpha: 1.1528532035454195
R-hat for nu: 10.488763895489186
Mean of alpha: 1.986241615455652
Mean of nu: 15.68712508354061
acceptance rate